# ACCESS-AIS3 -- Assemble inverted fields & SSA relaxation

## Imports & helper functions

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Configure options

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'


all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
]

# Define steps to run (this notebook is scoped to this group)
steps = ['ssa_inverted_solve', 'ssa_relaxation']

## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

## Initialise data catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Assemble inverted fields and solve stress balance - Full domain

In [ ]:
if 'ssa_inverted_solve' in steps:

    print("-------------------------------------------------------------")
    print(f" ASSEMBLING INVERTED MODEL AND SOLVING STRESS BALANCE"       )
    print("-------------------------------------------------------------")

    print(f"-- Loading parameterized model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')

    print(f"-- Updating rheology B from floating-ice rheology L-curve ({rheology_lcurve_run})...")
    mdr = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_rheology_floating_inv_lcurve/{rheology_lcurve_run}/{rheology_lcurve_run}.nc')
    md.materials.rheology_B[mdr.mesh.extractedvertices - 1] = mdr.results.StressbalanceSolution.MaterialsRheologyBbar # Note: -1 for zero-based indexing

    print(f"-- Updating friction field from grounded-ice friction L-curve ({friction_lcurve_run})...")
    mdf = pyissm.model.io.load_model(f'{model_dir}/AIS3_ssa_friction_inv_reg_lcurve/{friction_lcurve_run}/{friction_lcurve_run}.nc')
    fric_control, fric_field, _, _ = friction_law_info(md)  # Schoof: FrictionC / C ; Budd: FrictionCoefficient / coefficient
    _fld = getattr(md.friction, fric_field).astype(float)
    _grafted = np.asarray(getattr(mdf.results.StressbalanceSolution, fric_control)).ravel()
    if fric_control == 'FrictionC':
        # Schoof's saved C has literal 0 on floating ice (clamped there by the inversion's own
        # min/max_parameters=0), which fails check_consistency's "friction.C > 0.0" requirement.
        # Floor to a tiny positive value -- floating ice has N=0 too, so it's physically inert
        # regardless (same fix used throughout this session's scratchpad scripts).
        _grafted = np.maximum(_grafted, 0.01)
    _fld[mdf.mesh.extractedvertices - 1] = _grafted # -1 for zero-based indexing
    setattr(md.friction, fric_field, _fld)

    if fric_control == 'FrictionC':
        # BUGFIX: only the C *field* was being grafted -- md.friction.Cmax was left at
        # ais_0.1_param.py's own default (0.85, set fresh every 'param' run), NOT the Cmax
        # this C field was actually fit under (2.0, the validated tightrestol continuation
        # result). Confirmed as the cause of a second wave of extreme-velocity vertices
        # (up to ~57,000 m/yr) after the thin-ice pin fix resolved the first, catastrophic
        # wave: those vertices all showed Cmax=0.85 with substantial thickness/N, not the
        # thin-ice signature -- i.e. C values calibrated to be safe under a loose Cmax=2.0
        # ceiling suddenly hard-capped at a much tighter 0.85 ceiling they were never tuned
        # for. Set explicitly to match the fitted value.
        md.friction.Cmax = np.full(md.mesh.numberofvertices, 2.0)

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print(f"-- Flooring thin ice at 100m (numerical stability), preserving observed surface...")
    # Must match the geometry ssa_friction_inv_lcurve/ssa_friction_inv_reg_lcurve actually
    # solved against -- otherwise this assembled forward solve pairs the inverted friction
    # field with different driving stress than it was tuned for. See ssa_friction_inv_lcurve
    # for the full rationale.
    _ri = md.materials.rho_ice; _rw = md.materials.rho_water
    _H = np.asarray(md.geometry.thickness).ravel().copy()
    _H_orig = _H.copy()  # pre-floor thickness, used below to identify thin-ice vertices
    _ol = np.asarray(md.mask.ocean_levelset).ravel()
    _surf0 = np.asarray(md.geometry.surface).ravel().copy()
    _bed = np.asarray(md.geometry.bed).ravel()
    _H = np.maximum(_H, 100.0)
    _flt = _ol < 0
    _base = np.empty_like(_H); _surf = np.empty_like(_H)
    _base[_flt] = -_H[_flt] * _ri / _rw; _surf[_flt] = _H[_flt] * (1.0 - _ri / _rw)
    # BUGFIX: hydrostatic floating draft can occasionally sit below the local seafloor
    # (base < bed) -- a second, separate consistency failure caught by the same
    # ssa_relaxation check once the grounded base=bed fix above cleared the first one.
    # Clamp the floating base at bed as a numerical floor; leaves the ocean_levelset
    # floating/grounded classification untouched (not attempting a full grounding-line
    # reclassification here), just prevents the ice keel from being placed underground.
    _base[_flt] = np.maximum(_base[_flt], _bed[_flt])
    # Keep thickness=surface-base self-consistent wherever the clamp above actually moved
    # base (a no-op at every other floating vertex, since surf was already H*(1-ri/rw) there,
    # which equals base+H when base is the unclamped hydrostatic value).
    _surf[_flt] = _base[_flt] + _H[_flt]
    # BUGFIX: grounded base was previously set to (original surface - floored thickness),
    # which is only consistent with the true bed for vertices that weren't actually floored.
    # Anywhere thickness WAS floored (H_orig<100), this pushed base below the real bed by
    # exactly the amount added -- caught by ssa_relaxation's transient/masstransport
    # consistency check ("base < bed", "base=bed on grounded ice violated"), which
    # ssa_inverted_solve's own stress-balance-only solve never exercises. Grounded ice must
    # have base=bed by definition; the original "preserve observed surface" design was
    # actually incompatible with that once thickness is floored (bed is fixed real
    # topography, so raising thickness has to raise the surface instead). Surface only
    # changes from the observed value at the ~85,000 vertices that were already thin/floored;
    # everywhere else this is a no-op (surf0 - H_orig == bed there already).
    _base[~_flt] = _bed[~_flt]
    _surf[~_flt] = _bed[~_flt] + _H[~_flt]
    md.geometry.thickness = _H; md.geometry.base = _base; md.geometry.surface = _surf

    # Fix negative effective pressure (consistent with the inversion setup)
    _lim = 0.07
    N = md.friction.effective_pressure.copy()
    N[N < 0] = 0
    _Nfloor = _lim * md.materials.rho_ice * md.constants.g * _H
    N = np.maximum(N, _Nfloor)
    md.friction.effective_pressure = N
    md.friction.effective_pressure_limit = _lim

    if fric_control == 'FrictionC':
        # Scoped fix for a known, documented Schoof failure mode (docs/inversion_worklog.md
        # 5.4): at the thickness floor, N_floor = 0.07*rho_ice*g*H is weak (~6300 Pa at
        # H=100m), so Schoof's yield ceiling Cmax*N is only ~5000-12600 Pa -- nowhere near
        # enough to hold back driving stress on steep terrain. Unlike Budd (resistance keeps
        # growing, however weakly, with velocity, so Newton always finds a finite
        # equilibrium), Schoof's resistance is hard-capped: once driving stress exceeds the
        # cap there is no equilibrium at any finite velocity. First full-mesh attempt with
        # this Schoof result confirmed this exact signature: all 24 catastrophic-velocity
        # vertices (up to 3e19 m/yr) sat at/near the original (pre-floor) thickness floor,
        # inside the friction-inversion subdomain where this C field was fit. Raising the
        # floor doesn't fix this -- N_floor scales with H, so any floor value has the same
        # weak-yield-ceiling problem somewhere steep enough. Dirichlet-pinning these specific
        # vertices to observed velocity sidesteps the gap without touching the C field
        # elsewhere -- same pattern as this pipeline's existing constrain_Schoof.exp regions
        # and Felicity's own Inversion_Friction_Schoof approach.
        print(f"-- Pinning thin-ice vertices in the Schoof-fitted subdomain to observed velocity (known Coulomb-cap/thin-ice instability, see docs/inversion_worklog.md 5.4)...")
        _in_subdomain = np.zeros(md.mesh.numberofvertices, dtype=bool)
        _in_subdomain[mdf.mesh.extractedvertices - 1] = True
        _thin_orig = _H_orig <= 100.0
        _has_obs = np.asarray(md.inversion.vel_obs).ravel() > 0
        _vxo = np.asarray(md.inversion.vx_obs).ravel()
        _vyo = np.asarray(md.inversion.vy_obs).ravel()
        _pin = _thin_orig & _in_subdomain & _has_obs
        md.stressbalance.spcvx[_pin] = _vxo[_pin]
        md.stressbalance.spcvy[_pin] = _vyo[_pin]
        md.stressbalance.spcvz[_pin] = 0
        print(f"   pinned {int(_pin.sum())} thin-ice vertices ({int((_thin_orig&_in_subdomain).sum())} thin-and-in-subdomain, {int((_thin_orig&_in_subdomain&~_has_obs).sum())} had no observation to pin to)")

        # Second, separate wave of extreme velocities (up to ~57,000 m/yr) survived the
        # thin-ice pin above and the Cmax fix. Investigated directly: not yield-cap-limited
        # (driving stress exceeded the Coulomb ceiling at only 7 of 835 extreme vertices),
        # not a solver-tolerance artifact (identical result at residue_threshold=1e-3 and
        # 1e-5), and not really a sharp bound-clamping discontinuity either (only 2 of 835
        # showed that specific pattern). What the remaining 833 do share: 715 of them sit
        # exactly at the lower C bound (50) -- the inversion wanted less resistance than the
        # bound allowed, over broad contiguous patches, not a localised artifact. Rather than
        # chase the exact numerical mechanism further, extend the same Dirichlet-pin approach
        # (safe by construction: pins to the actual observation, never wrong) to all grounded
        # lower-bound-clamped vertices in the subdomain, not just the ~717 that happened to
        # blow up this time -- the other ~84,700 don't misbehave now, but nothing guarantees
        # that under a different warm-start/Cmax step.
        print(f"-- Pinning grounded lower-bound-clamped (C<=50) vertices in the Schoof-fitted subdomain to observed velocity (second wave of extreme velocities, mechanism not fully isolated -- see comment above)...")
        # BUGFIX: C<=50.01 alone also matched floating ice (floored to C=0.01, correctly, not
        # an instability signature) -- first attempt pinned 310,476 vertices instead of the
        # intended ~85,000 grounded ones, silently replacing floating-shelf physics with
        # observations everywhere. Restricted to grounded (~_flt) to match original intent.
        _at_lower_bound = _fld <= 50.01
        _pin2 = _at_lower_bound & _in_subdomain & _has_obs & (~_pin) & (~_flt)
        md.stressbalance.spcvx[_pin2] = _vxo[_pin2]
        md.stressbalance.spcvy[_pin2] = _vyo[_pin2]
        md.stressbalance.spcvz[_pin2] = 0
        print(f"   pinned {int(_pin2.sum())} additional lower-bound-clamped vertices")

        # Third wave: even after both pins above, the worst remaining offenders (by far the
        # highest velocities, up to ~35,000 m/yr) are overwhelmingly classified ice_levelset
        # >= 0 -- i.e. NOT ice at all. These are ice-front-ring vertices pulled into the
        # friction-inversion subdomain by its element-level ice_levelset_elements<1 inclusion
        # criterion (deliberately includes elements straddling the true margin), which can
        # include individual vertices on the non-ice side of that boundary. They still
        # received a fitted, non-floor C value from the graft, despite there being no real
        # ice there to have meaningful velocity in the first place -- neither the thin-ice nor
        # the lower-bound-C criterion reliably catches them (thickness/C at these vertices
        # doesn't consistently fall in either range). Pin directly on the ice_levelset>=0
        # classification instead.
        print(f"-- Pinning non-ice-classified (ice_levelset>=0) vertices in the Schoof-fitted subdomain to observed velocity (third wave, worst remaining offenders)...")
        _ice_full = np.asarray(md.mask.ice_levelset).ravel()
        _nonice = _ice_full >= 0
        _pin3 = _nonice & _in_subdomain & _has_obs & (~_pin) & (~_pin2)
        md.stressbalance.spcvx[_pin3] = _vxo[_pin3]
        md.stressbalance.spcvy[_pin3] = _vyo[_pin3]
        md.stressbalance.spcvz[_pin3] = 0
        print(f"   pinned {int(_pin3.sum())} additional non-ice-classified vertices")

    print(f"-- Disabling inversion (forward solve only)...")
    md.inversion.iscontrol = 0
    md.verbose.solution = 1

    print(f"-- Assigning cluster and updating settings...")
    md.miscellaneous.name = 'AIS3_inverted'
    md.cluster = cluster
    # BUGFIX: this step previously set waitonlock=0 (disables blocking-wait) but then called
    # solve() with load_only=True, which never submits a job -- it only loads results from an
    # already-finished prior run (pyissm/model/execute.py:1078-1082, unconditional early
    # return). There is no earlier load_only=False submission anywhere in this step, so it has
    # never actually completed a real run before (confirmed: "Binary file AIS3_inverted.outbin
    # not found" on the first real attempt). Fixed to the single-call synchronous pattern:
    # load_only=False actually submits, and waitonlock>0 makes solve() block internally,
    # polling for the cluster job's lock file, then auto-load results in the same call.
    md.settings.waitonlock = 120  # minutes
    # Every other step in this pipeline sets this explicitly to 1e-3; ssa_inverted_solve was
    # the one exception, silently left at ISSM's much tighter 1e-6 default. First real attempt
    # hit "solver residue too high! norm(KU-F)/norm(F)=1.09e-06 > 1e-06" (marginal, ~9% over)
    # on the very first linear solve, cascading into "Recovery solver failed" across many MPI
    # ranks. 1e-3 (matching every other step) cleared that crash but is a 1000x loosening for
    # a failure that was only ~9% over the original threshold -- traced as the likely cause of
    # a second-wave problem: after the thin-ice pin and Cmax fixes, ~835 vertices scattered
    # across many unrelated locations (not one cluster) still showed spurious velocities up to
    # ~57,000 m/yr, but only 7 of them actually had driving stress exceeding the Schoof yield
    # ceiling -- ruling out yield-cap physics and pointing at under-converged local solutions
    # slipping through the loosened tolerance instead. Tightened to 1e-5, a much smaller step
    # down from the 1e-6 default, to test whether that's enough to clear the original marginal
    # failure without opening the door this wide.
    md.settings.solver_residue_threshold = 1e-5

    if save:
        print(f"-- Submitting and waiting on stress balance solution...")
        md = pyissm.model.execute.solve(md, 'Stressbalance', load_only = False, runtime_name = False)

        if diagnostics:
            vel = md.results.StressbalanceSolution.Vel
            vel_obs = md.inversion.vel_obs
            residual = vel - vel_obs
            print(f"\nFORWARD SOLVE DIAGNOSTICS:")
            print(f"   Max modelled velocity: {np.nanmax(vel):.2f} m/yr")
            print(f"   Velocity RMSE vs obs:  {np.sqrt(np.nanmean(residual**2)):.2f} m/yr")

        if plot:
            # BUGFIX: plot_model_field returns (fig, ax, trip) when ax isn't passed in
            # (pyissm/plot/plot.py:562), not (fig, ax) as its own docstring claims -- this
            # step never reached this line successfully before to catch it.
            fig, ax, _trip = pyissm.plot.plot_model_field(md, md.results.StressbalanceSolution.Vel,
                                                   show_cbar = True,
                                                   cmap = 'PuOr',
                                                   cbar_kwargs = {'label': 'Modelled velocity (m/a)'})
            ax.set_title('Inverted model - SSA stress balance velocity')
            plt.savefig(f'{model_dir}/AIS3_inverted_velocity.png')

        print(f"\nSaving inverted model to {model_dir}/AIS3_inverted.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_inverted.nc')

    else:
        print(f"-- Submitting stress balance solve...")
        md = pyissm.model.execute.solve(md, 'Stressbalance', load_only = False, runtime_name = False)

## Transient relaxation - Full domain

In [ ]:
if 'ssa_relaxation' in steps:

    print("-------------------------------------------------------------")
    print(f" TRANSIENT RELAXATION"                                       )
    print("-------------------------------------------------------------")

    print(f"-- Loading inverted model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_inverted.nc')

    print(f"-- Configuring transient relaxation...")
    md.inversion.iscontrol = 0
    md.verbose.solution = 1

    # Relax the dynamics + free surface to damp initialisation shock.
    # Thermal and SMB are held fixed; grounding line is allowed to migrate.
    md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
    md.transient.isstressbalance = 1
    md.transient.ismasstransport = 1
    md.transient.issmb = 1
    md.transient.isthermal = 0
    md.transient.isgroundingline = 1
    md.groundingline.migration = 'SubelementMigration'
    md.transient.requested_outputs = ['default', 'Vel', 'Thickness', 'Surface', 'Base', 'MaskOceanLevelset']

    # BUGFIX: masstransport.spcthickness defaults to a bare scalar NaN
    # (pyissm/model/classes/masstransport.py:62), not a properly-shaped per-vertex array --
    # never caught before because ismasstransport was never actually enabled until this step.
    # NaN still means "no constraint" per that class's own convention; just needs the right
    # shape.
    md.masstransport.spcthickness = np.full(md.mesh.numberofvertices, np.nan)

    # TODO: no real SMB dataset is wired into this pipeline anywhere yet (ais_0.1_param.py
    # never sets md.smb.mass_balance) -- issmb was never actually enabled until this step, so
    # this gap was never caught before. Zero SMB is a defensible placeholder for THIS short
    # (20yr) diagnostic relaxation, whose purpose is damping initialisation shock in the
    # dynamics/geometry, not simulating real mass-balance evolution -- but a later stage that
    # needs actual climate forcing (e.g. historical_dhdt_tuning) must not inherit this as-is;
    # see pyissm/model/classes/smb.py's own "not specified -- set to 0" fallback (never fires
    # before this step's own consistency check runs, hence set explicitly here).
    md.smb.mass_balance = np.zeros(md.mesh.numberofvertices)

    # Same NaN-default gap as smb.mass_balance above, this time in basalforcings (also never
    # exercised before ismasstransport was first enabled here): groundedice_melting_rate and
    # floatingice_melting_rate both default to scalar NaN
    # (pyissm/model/classes/basalforcings.py:50-51), causing a marshalling-time crash
    # ("FetchDataToInput ... not found in binary file") rather than the Python-level
    # consistency error the other gaps hit. Zero basal melt is the standard placeholder absent
    # real ocean-forcing data (see melt_gamma_tuning stage below for where that eventually
    # belongs) and matches this class's own "not specified -- set to 0" fallback intent.
    md.basalforcings.groundedice_melting_rate = np.zeros(md.mesh.numberofvertices)
    md.basalforcings.floatingice_melting_rate = np.zeros(md.mesh.numberofvertices)

    # Short relaxation window -- TODO: tune final_time / time_step for your domain.
    md.timestepping.start_time = 0
    md.timestepping.final_time = 20    # years
    md.timestepping.time_step  = 0.05  # years

    print(f"-- Assigning cluster and updating settings...")
    md.miscellaneous.name = 'AIS3_relaxed'
    md.cluster = cluster
    # BUGFIX: same pattern found and fixed in ssa_inverted_solve -- waitonlock=0 disables the
    # blocking wait, but the "if save" branch below calls solve() with load_only=True, which
    # never submits (pyissm/model/execute.py:1078-1082 unconditional early return); it only
    # loads results from an already-finished prior run. The correct load_only=False call
    # already exists in the "else" branch below, just gated behind save=False -- since save is
    # True globally in this script, that branch was unreachable and this step has likely never
    # actually completed. Fixed to the single-call synchronous pattern used in
    # ssa_inverted_solve: load_only=False submits, waitonlock>0 blocks internally polling for
    # the lock file, then auto-loads in the same call. 24h budget for a 400-timestep transient
    # (20yr / 0.05yr step) -- generous given a single forward solve took ~3-5min.
    md.settings.waitonlock = 1440  # minutes
    # Also missing from this step (present in every other step in this pipeline, including the
    # now-fixed ssa_inverted_solve): the tighter default 1e-6 residue threshold caused a
    # marginal-failure crash there ("Recovery solver failed") on the very first linear solve;
    # a 400-timestep transient has far more exposure to that same marginal-residual failure.
    md.settings.solver_residue_threshold = 1e-3

    if save:
        print(f"-- Submitting and waiting on transient relaxation...")
        md = pyissm.model.execute.solve(md, 'Transient', load_only = False, runtime_name = False)

        if diagnostics:
            dH = md.results.TransientSolution.Thickness[-1] - md.geometry.thickness
            print(f"\nRELAXATION DIAGNOSTICS:")
            print(f"   Max |dH| over relaxation: {np.nanmax(np.abs(dH)):.2f} m")
            print(f"   Mean |dH| over relaxation: {np.nanmean(np.abs(dH)):.2f} m")

        print(f"\nSaving relaxed model to {model_dir}/AIS3_relaxed.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_relaxed.nc')

    else:
        print(f"-- Submitting transient relaxation...")
        md = pyissm.model.execute.solve(md, 'Transient', load_only = False, runtime_name = False)